# Logistic regression and a feedforward neural network

[Open in Colab](https://colab.research.google.com/github/aburkov/theLMbook/blob/main/linear_and_fnn_models.ipynb)

This chapter 1 example compares two binary classifiers on synthetic real-estate
data. Each row contains the floor area and the number of bedrooms. Each label
belongs to the corresponding row.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn

torch.manual_seed(42)

features = torch.tensor([
    [1200, 3],
    [850, 2],
    [1500, 4],
    [700, 1],
    [1100, 3],
    [950, 2],
    [1800, 4],
    [600, 1],
    [1300, 3],
    [1000, 2],
    [2000, 5],
    [1000, 4],
], dtype=torch.float32)
labels = torch.tensor([0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0], dtype=torch.float32).unsqueeze(1)

# Put area and bedroom count on comparable scales.
features = (features - features.mean(dim=0)) / features.std(dim=0, unbiased=False)

Logistic regression uses one affine layer with two weights and one bias.
The feedforward network adds a hidden layer of ten units and a sigmoid activation.
The models produce logits. During training, binary cross-entropy with logits
applies the final sigmoid internally for numerical stability.

In [ ]:
logistic_model = nn.Sequential(nn.Linear(2, 1))
feedforward_model = nn.Sequential(
    nn.Linear(2, 10),
    nn.Sigmoid(),
    nn.Linear(10, 1),
)

def train(model, inputs, targets, steps=500):
    optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
    loss_function = nn.BCEWithLogitsLoss()
    history = []

    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_function(model(inputs), targets)
        loss.backward()
        optimizer.step()
        history.append(loss.item())

    return history

logistic_loss = train(logistic_model, features, labels)
feedforward_loss = train(feedforward_model, features, labels)

In [ ]:
plt.plot(logistic_loss, label="Logistic regression")
plt.plot(feedforward_loss, label="Feedforward neural network")
plt.xlabel("Training step")
plt.ylabel("Binary cross-entropy")
plt.legend()
plt.show()

Both curves measure training loss. Comparing generalization requires a separate evaluation set.